# Example 18 — Link-Level Simulator: BER/SER/PER Curves

**Level:** Advanced

After working through this notebook you will know how to:

- Access reference constellations from the Rust backend
- Use `CoherentReceiver` to demodulate RRC-filtered waveforms
- Compute BER, SER, and PER with the new metric functions
- Run a `LinkSimulator` Eb/N0 sweep for BER curve generation
- Compare simulated BPSK BER against the theoretical Q-function curve
- Compare BER performance across modulation schemes

In [ ]:
%matplotlib inline

import sys

sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
from spectra._rust import (
    apply_rrc_filter_with_taps,
    generate_bpsk_symbols_with_indices,
    get_bpsk_constellation,
    get_qam_constellation,
    get_qpsk_constellation,
)
from spectra.link import LinkSimulator
from spectra.metrics import bit_error_rate
from spectra.receivers import CoherentReceiver
from spectra.receivers.coherent import constellation_to_bits
from spectra.utils.rrc_cache import cached_rrc_taps
from spectra.waveforms import BPSK, QAM16, QPSK

# ── Configuration ──────────────────────────────────────────────────────────────
SEED         = 42
NUM_SYMBOLS  = 10000
PACKET_LEN   = 200
EB_N0_RANGE  = np.arange(0, 14, 1.0)

## 1. Constellations

SPECTRA's Rust backend exposes reference constellation point generators for each modulation: `get_bpsk_constellation()`, `get_qpsk_constellation()`, and `get_qam_constellation(M)`. These return the ideal complex-valued symbol points used both for modulation and for minimum-distance decision in the `CoherentReceiver`.

Plotting the constellations is a quick sanity check: BPSK has 2 antipodal points on the real axis, QPSK has 4 points on the unit circle at 45° offsets, and 16-QAM has a 4×4 grid of 16 points. The integer label annotated beside each point is its symbol index, which maps to a Gray-coded bit pattern inside the receiver.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (label, get_const) in zip(axes, [
    ("BPSK", get_bpsk_constellation),
    ("QPSK", get_qpsk_constellation),
    ("16QAM", lambda: get_qam_constellation(16)),
]):
    const = np.array(get_const())
    ax.scatter(const.real, const.imag, s=80, c="steelblue", zorder=5)
    for i, pt in enumerate(const):
        ax.annotate(str(i), (pt.real, pt.imag), textcoords="offset points",
                    xytext=(6, 6), fontsize=7, color="gray")
    ax.set_title(f"{label} Constellation")
    ax.set_xlabel("I")
    ax.set_ylabel("Q")
    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 2. Noiseless Demodulation

`CoherentReceiver` implements a matched-filter receive chain: it applies a matched RRC filter to the incoming IQ samples, downsamples to one sample per symbol, and performs minimum-distance symbol decisions against the reference constellation.

In the noiseless case (no AWGN added) the BER should be exactly 0 after trimming the filter transient edges. The `filter_span` attribute of each waveform specifies how many symbol periods the RRC filter's group-delay spans on each side; trimming those edge symbols removes the startup and tail transients that would otherwise produce false errors.

In [ ]:
wf = BPSK(samples_per_symbol=8)
symbols, tx_indices = generate_bpsk_symbols_with_indices(500, SEED)
taps = cached_rrc_taps(wf.rolloff, wf.filter_span, wf.samples_per_symbol)
tx_iq = np.array(apply_rrc_filter_with_taps(symbols, taps, wf.samples_per_symbol))

rx = CoherentReceiver(wf)
rx_indices, rx_bits = rx.demodulate(tx_iq)

# Compare within valid region (trim filter transients)
trim = wf.filter_span
tx_arr = np.array(tx_indices)
valid_tx = tx_arr[trim:trim + len(rx_indices) - trim]
valid_rx = rx_indices[trim:trim + len(valid_tx)]

ber_noiseless = bit_error_rate(
    constellation_to_bits(valid_tx, 2),
    constellation_to_bits(valid_rx, 2),
)
print(f"Noiseless BPSK demodulation BER: {ber_noiseless:.6f}")

## 3. BER/SER/PER Metrics

SPECTRA provides three complementary link-quality metrics:

- **BER (Bit Error Rate):** fraction of individual bits received in error. This is the most granular measure and is independent of how bits are grouped into symbols or packets.
- **SER (Symbol Error Rate):** fraction of symbols received in error. For BPSK (1 bit/symbol) SER = BER; for higher-order modulations a single symbol error can corrupt multiple bits, so SER ≥ BER.
- **PER (Packet Error Rate):** fraction of fixed-length packets (of `packet_length` bits) that contain at least one bit error. PER is always ≥ BER and grows quickly with packet length because a longer packet has more opportunities to collect at least one error.

`LinkSimulator` wraps all three into a single `run(eb_n0_db_array)` call that returns a `LinkResults` object.

In [ ]:
# Simulate at a single Eb/N0 point to show all three metrics
sim = LinkSimulator(waveform=BPSK(samples_per_symbol=8),
                    num_symbols=5000, packet_length=PACKET_LEN, seed=SEED)
result_single = sim.run(np.array([4.0]))

print("BPSK at Eb/N0 = 4 dB:")
print(f"   BER = {result_single.ber[0]:.4e}")
print(f"   SER = {result_single.ser[0]:.4e}")
print(f"   PER = {result_single.per[0]:.4e} (packet_length={PACKET_LEN})")

## 4. BPSK BER vs. Theoretical

For BPSK in AWGN the theoretical BER is given exactly by the Q-function:

$$\text{BER}_\text{BPSK} = Q\!\left(\sqrt{2\,E_b/N_0}\right)$$

where $Q(x) = \frac{1}{2}\,\text{erfc}\!\left(\frac{x}{\sqrt{2}}\right)$.

`LinkResults.theoretical_ber()` returns this curve when the waveform supports it (returns `None` otherwise). Comparing the simulated curve to the theoretical one validates both the transmitter chain (RRC pulse shaping) and the receiver (matched filter + symbol decision). Good agreement requires enough symbols — here we use 10 000 — to drive down Monte Carlo variance at low-error-rate operating points.

In [ ]:
sim_bpsk = LinkSimulator(waveform=BPSK(samples_per_symbol=8),
                         num_symbols=NUM_SYMBOLS, seed=SEED)
results_bpsk = sim_bpsk.run(EB_N0_RANGE)

theory = results_bpsk.theoretical_ber()

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(results_bpsk.eb_n0_db, results_bpsk.ber, "o-", color="steelblue",
            label="Simulated BER", markersize=5)
if theory is not None:
    ax.semilogy(results_bpsk.eb_n0_db, theory, "k--", label="Theoretical BER", linewidth=1.5)
ax.set_xlabel("Eb/N0 (dB)")
ax.set_ylabel("Bit Error Rate")
ax.set_title(f"BPSK BER — Simulated vs. Theoretical ({NUM_SYMBOLS} symbols)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
ax.set_ylim(bottom=1e-5)
plt.tight_layout()
plt.show()

## 5. Multi-Modulation Comparison

Higher-order modulations pack more bits per symbol, which increases spectral efficiency, but they also require a higher Eb/N0 to achieve the same BER because the constellation points are closer together relative to the signal power.

- **BPSK** (1 bit/symbol): most robust — largest minimum distance for a given average power.
- **QPSK** (2 bits/symbol): same BER as BPSK when measured against Eb/N0 (both use antipodal pairs in orthogonal dimensions), but doubles throughput.
- **16-QAM** (4 bits/symbol): needs roughly 4–6 dB more Eb/N0 than BPSK/QPSK to reach the same BER floor because its 16 tightly packed points are more susceptible to noise.

The waterfall curves below illustrate this tradeoff clearly on a log-scale BER axis.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {"BPSK": "steelblue", "QPSK": "coral", "16QAM": "seagreen"}

for wf_cls, label in [(BPSK, "BPSK"), (QPSK, "QPSK"), (QAM16, "16QAM")]:
    sim = LinkSimulator(waveform=wf_cls(samples_per_symbol=8),
                        num_symbols=NUM_SYMBOLS, seed=SEED)
    results = sim.run(EB_N0_RANGE)
    # Replace zero BER with NaN so they don't appear on log plot
    ber_plot = results.ber.copy()
    ber_plot[ber_plot == 0] = np.nan
    ax.semilogy(results.eb_n0_db, ber_plot, "o-", color=colors[label],
                label=label, markersize=5)

ax.set_xlabel("Eb/N0 (dB)")
ax.set_ylabel("Bit Error Rate")
ax.set_title(f"BER Comparison — BPSK vs. QPSK vs. 16QAM ({NUM_SYMBOLS} symbols)")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
ax.set_ylim(bottom=1e-5)
plt.tight_layout()
plt.show()

## 6. SER and PER Curves

**SER vs. BER:** For QPSK each symbol carries 2 bits. A symbol error always corrupts at least one bit, but with Gray coding it usually corrupts exactly one bit. As a result SER ≈ BER at high Eb/N0 but SER can be up to twice BER at low Eb/N0 where multi-bit symbol errors become more likely.

**PER dependence on packet length:** A packet of length $L$ bits fails if *any* of its bits are in error. Under the independence assumption:

$$\text{PER} = 1 - (1 - \text{BER})^L$$

For $L = 200$ bits and BER = $10^{-2}$, PER ≈ $1 - 0.99^{200} \approx 0.87$ — most packets are corrupted even at a seemingly low BER. This motivates forward-error-correction (FEC) coding to reduce the effective BER before the packet layer.

In [ ]:
sim_full = LinkSimulator(waveform=QPSK(samples_per_symbol=8),
                         num_symbols=NUM_SYMBOLS, packet_length=PACKET_LEN, seed=SEED)
results_full = sim_full.run(EB_N0_RANGE)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# SER
ser_plot = results_full.ser.copy()
ser_plot[ser_plot == 0] = np.nan
axes[0].semilogy(results_full.eb_n0_db, ser_plot, "s-", color="coral", markersize=5)
axes[0].set_xlabel("Eb/N0 (dB)")
axes[0].set_ylabel("Symbol Error Rate")
axes[0].set_title("QPSK Symbol Error Rate")
axes[0].grid(True, which="both", alpha=0.3)

# PER
per_plot = results_full.per.copy()
per_plot[per_plot == 0] = np.nan
axes[1].semilogy(results_full.eb_n0_db, per_plot, "D-", color="seagreen", markersize=5)
axes[1].set_xlabel("Eb/N0 (dB)")
axes[1].set_ylabel("Packet Error Rate")
axes[1].set_title(f"QPSK Packet Error Rate (L={PACKET_LEN} bits)")
axes[1].grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()